In [1]:
import os
import json
import numpy as np
import random
import time
import re
import tiktoken
from google import genai
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from openai import OpenAI


# # --- Hugging Face mirror/cache settings (set before loading embedding models) ---
# os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
# os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface"))

# print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
# print("HUGGINGFACE_HUB_CACHE:", os.environ.get("HUGGINGFACE_HUB_CACHE"))

In [2]:

# --- Configuration ---
EMBEDDING_DIM = 384  # Dimension for question and answer embeddings

UPDATE_FREQUENCY = 1    # Update JSON records after every question


# TODO: configure more parameters here
# Regularization parameter λ, exploration coefficient γ, step size η, network width m, 
# gradient descent steps J, LLM pool size M , batch size B_batch


REG = 1 #正则化参数
M = 7 # LLM pool size
GEMMA = 2 # 探索系数
NODE_JUDGE_THRESHOLD = 0.5 # 节点即时打分低于该阈值时触发回溯
MAX_NODE_BACKTRACKS = 2 # 每个节点最多回溯次数


In [3]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    # Notebook 常在 Algorithms/ 目录运行，.env 在项目根目录
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for p in candidates:
        if p.exists():
            dotenv_path = str(p)
            break
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print("Loaded .env:", dotenv_path)
else:
    print("No .env found; please create one at project root.")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")

Loaded .env: e:\code\321\DAG_Router_demo-gui\.env


In [4]:
# --- Test two models via OpenRouter ---
Decompose_MODEL_NAME = "deepseek/deepseek-v3.2"
GRADER_MODEL_NAME = "deepseek/deepseek-v3.2"

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY 未设置，请检查 .env"
base_url = OPENROUTER_BASE_URL 

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)

models_to_test = [Decompose_MODEL_NAME, GRADER_MODEL_NAME]
prompt = "请只回复：OK"

for model_name in models_to_test:
    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=16,
            temperature=0.0,
        )
        text = resp.choices[0].message.content if resp.choices else "<no choice>"
        print(f"✅ {model_name} -> {text}")
    except Exception as e:
        print(f"❌ {model_name} failed: {type(e).__name__}: {e}")

✅ deepseek/deepseek-v3.2 -> OK
✅ deepseek/deepseek-v3.2 -> OK


In [5]:
# 从项目根目录的 model_config.py 加载模型列表
import sys
from pathlib import Path

# 当前 notebook 在 Algorithm/ 下，根目录是其上一级
project_root = Path.cwd() if (Path.cwd() / "model_config.py").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model_config import MODELS_CONFIG

AVAILABLE_LLMS = list(MODELS_CONFIG.keys())
LLM_ID_DIM = len(AVAILABLE_LLMS)

print(f"Loaded {LLM_ID_DIM} models from model_config.py")
print(AVAILABLE_LLMS[:7])

Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct', 'qwen/qwen3.5-flash-02-23', 'mistralai/mistral-nemo']


In [6]:
import json
import os
import numpy as np
from pathlib import Path

# ==========================================
# TODO: load train set data/final_evaluation_dataset.json
# ==========================================
def load_dataset(file_path="../data/final_evaluation_dataset_500.json"):
    candidates = [
        Path(file_path),
        Path.cwd() / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "final_evaluation_dataset_500.json",
        Path.cwd() / "data" / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "data" / "final_evaluation_dataset_500.json",
    ]
    real_path = next((p for p in candidates if p.exists()), None)
    if real_path is None:
        print(f"⚠️ 警告: 数据集文件不存在。已尝试: {[str(p) for p in candidates]}")
        return []
    with open(real_path, 'r', encoding='utf-8') as f:
        print(f"✅ 成功加载数据集: {real_path}")
        return json.load(f)


def _to_jsonable(obj):
    """Recursively convert numpy / non-JSON-native objects to JSON-safe types."""
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

# ==========================================
# TODO: 创建记录模型回答的json文件
# ==========================================
class ResultRecorder:
    def __init__(self, output_file="execution_records.json"):
        # 仅使用“已有”的 record 目录（优先当前目录，其次上一级目录）
        candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
        record_dir = next((p for p in candidates if p.exists() and p.is_dir()), None)
        if record_dir is None:
            raise FileNotFoundError("未找到已有的 record 目录，请先创建 record 文件夹。")

        self.output_file = str(record_dir / Path(output_file).name)
        self.records = {}

        # 若记录文件已存在则覆盖；不存在则新建空文件
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(self.records, f, ensure_ascii=False, indent=2)

    def add_record(self, problem_id: str, question: str, expected_answers: str, 
                   final_status: str, attempts: list):
        """
        按照规定的字典格式写入单条测试记录。
        attempts 应该是一个包含 step, task_desc, chosen_model, reward, output, llm_input_messages 的列表字典。
        """
        self.records[problem_id] = {
            "question": question,
            "answers": expected_answers,
            "final_status": final_status,
            "steps_taken": len(attempts),
            "attempts": _to_jsonable(attempts)
        }
        self.save()

    def save(self):
        """将记录持久化到 JSON 文件中，支持 UPDATE_FREQUENCY"""
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(_to_jsonable(self.records), f, ensure_ascii=False, indent=2)

In [7]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
# TODO: choose an embedding model that can process long contexts.

In [8]:
import torch

# 自动选择 GPU；若不可用则回退到 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

class FeatureExtractor:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model.to(device)
        print("Embedding model loaded successfully.")

    def get_feature_vector(self, text_context):
        # 如果输入为空，返回全0向量（防止报错）
        if not text_context or text_context.strip() == "":
            return np.zeros(EMBEDDING_DIM, dtype=np.float32)
            
        vector = self.model.encode(
            text_context, 
            convert_to_numpy=True, 
            normalize_embeddings=True
        )
        return vector.astype(np.float32)

# 实例化特征提取器
extractor = FeatureExtractor()

Using device: cuda
Loading embedding model: BAAI/bge-small-en-v1.5...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1007)')))"), '(Request ID: 3d339edb-424f-4987-9f3a-96fbddd35909)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Embedding model loaded successfully.


In [9]:
# TODO
# 将上下文拼接好后再转换为384维向量
class FeatureExtractor:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model.to(device)
        print("Embedding model loaded successfully.")

    def get_feature_vector(self, text_context):
        # 如果输入为空，返回全0向量（防止报错）
        if not text_context or text_context.strip() == "":
            return np.zeros(EMBEDDING_DIM, dtype=np.float32)
            
        vector = self.model.encode(
            text_context, 
            convert_to_numpy=True, 
            normalize_embeddings=True
        )
        return vector.astype(np.float32)

# 实例化特征提取器
extractor = FeatureExtractor()

Loading embedding model: BAAI/bge-small-en-v1.5...
Embedding model loaded successfully.


In [10]:
class DAGNode:
    def __init__(self, node_id, task_description):
        self.node_id = node_id
        self.task_description = task_description
        self.predecessors = []
        self.successors = []
        self.layer = 0
        
        # 运行时状态
        self.input_context = ""
        self.output_content = ""      # 模型原始输出（可含思考过程）
        self.answer_content = ""      # 仅保留 <answer>...</answer> 提取结果
        self.selected_model = None
        self.feature_vector = None 
        self.reward = 0.0          

    def add_successor(self, node):
        if node not in self.successors:
            self.successors.append(node)
        if self not in node.predecessors:
            node.predecessors.append(self)

class DAGGraph:
    def __init__(self):
        self.nodes = {}

    def add_node(self, node_id, description):
        if node_id not in self.nodes:
            self.nodes[node_id] = DAGNode(node_id, description)
        return self.nodes[node_id]

    def add_edge(self, source_id, target_id):
        if source_id in self.nodes and target_id in self.nodes:
            self.nodes[source_id].add_successor(self.nodes[target_id])

    def compute_layers(self):
        #层级计算：Layer = max(predecessor_layers) + 1
        for node in self.nodes.values(): node.layer = 0
        for _ in range(len(self.nodes) + 1):
            for node in self.nodes.values():
                for pred in node.predecessors:
                    if node.layer < pred.layer + 1:
                        node.layer = pred.layer + 1
        layers = {}
        for node in self.nodes.values():
            if node.layer not in layers: layers[node.layer] = []
            layers[node.layer].append(node)
        return layers


In [11]:
import json
import re


def decompose_query_to_dag(query_text: str, problem_id: str, client) -> DAGGraph:
    """
    Algorithm 1 Line 4: Decompose q_t into a DAG G_t
    Calls the LLM to decompose the original natural language query into a standardized DAG JSON format.

    If decomposition fails:
    - print error info
    - fallback to a single node (directly answer original question)
    """

    system_prompt = """
    You are an expert in logical task decomposition. Please decompose the user's complex problem into a Directed Acyclic Graph (DAG) of derivation steps.

    CRITICAL RULES:
    1. You are a planner, NOT a solver: Do NOT directly compute answers, simplify equations, or solve the problem in the task descriptions.
    2. Context preservation: Ensure each sub-task description explicitly includes all information it needs from the original problem statement.
    3. Dependency sufficiency: When assigning predecessors for a node, ensure those predecessors can provide ALL required information, physical quantities, and mathematical variables for the current sub-task.
    4. Multiple-choice questions: If the original problem has options, include full options in the final selection node.
    5. Execution output rule: In each node description, require the solver to wrap the final concise answer with <answer>...</answer>.
    6. If the Original Question includes explicit multiple-choice options (e.g., A, B, C, D), create a final node to match the extracted answer to the correct option letter. If the Original Question is open-ended and does NOT contain options, DO NOT hallucinate options. 
    The final node MUST instruct the model to output the exact entity, number, or option required by the Original Question.

    Output MUST be valid JSON only, with this structure (dependencies are written per node, NOT via edges):
    {
      "nodes": [
        {
          "id": "1",
          "description": "Extract known values from the problem and write them clearly. Put final concise result in <answer>...</answer>.",
          "predecessors": []
        },
        {
          "id": "2",
          "description": "Using values from Node 1, apply the required formula to compute the target quantity. Put final concise result in <answer>...</answer>.",
          "predecessors": ["1"]
        },
        {
          "id": "3",
          "description": "Match the computed result from Node 2 to the full options list (A..., B..., ...). Output ONLY the option letter in <answer>...</answer>.",
          "predecessors": ["2"]
        }
      ]
    }
    """

    def _normalize_task_desc(desc: str) -> str:
        return (desc or "").strip()

    def _fallback_graph(err_msg: str) -> DAGGraph:
        print(f"❌ Decomposition failed: {err_msg}")
        g = DAGGraph()
        g.problem_description = query_text
        g.add_node(
            "fallback_node",
            _normalize_task_desc(
                "Answer the following question directly and wrap final concise answer in <answer>...</answer>: " + query_text
            ),
        )
        return g

    user_prompt = f"Problem ID: {problem_id}\nOriginal Question: {query_text}"

    # 1) Call decomposition model and parse JSON
    try:
        resp = client.chat.completions.create(
            model=Decompose_MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=1400,
            temperature=0.2,
        )
        raw_output = (resp.choices[0].message.content or "").strip()

        json_str = raw_output
        if "```json" in json_str:
            json_str = json_str.split("```json", 1)[1].split("```", 1)[0].strip()
        elif "```" in json_str:
            json_str = json_str.split("```", 1)[1].split("```", 1)[0].strip()

        dag_dict = json.loads(json_str)
    except Exception as e:
        return _fallback_graph(f"model call / JSON parse error: {type(e).__name__}: {e}")

    # 2) Build graph (兼容新旧 schema)
    try:
        graph = DAGGraph()
        graph.problem_description = dag_dict.get("problem_description", query_text)

        nodes_data = dag_dict.get("nodes", [])
        if not isinstance(nodes_data, list) or len(nodes_data) == 0:
            raise ValueError("'nodes' missing or empty")

        # 新 schema: id/description/predecessors
        schema_new = all(
            isinstance(n, dict) and ("id" in n) and ("description" in n) and ("predecessors" in n)
            for n in nodes_data
        )
        # 兼容 schema A: node_id/sub_task_description/dependency_node_id
        schema_a = all(isinstance(n, dict) and ("node_id" in n) and ("sub_task_description" in n) for n in nodes_data)
        # 兼容 schema B: id/description + edges
        schema_b = all(isinstance(n, dict) and ("id" in n) and ("description" in n) for n in nodes_data)

        if schema_new:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)

            for n_data in nodes_data:
                target_id = str(n_data["id"])
                preds = n_data.get("predecessors", [])
                if not isinstance(preds, list):
                    preds = []
                for source_id in preds:
                    graph.add_edge(str(source_id), target_id)

        elif schema_a:
            for n_data in nodes_data:
                node_id = str(n_data["node_id"])
                task_desc = _normalize_task_desc(str(n_data["sub_task_description"]))
                graph.add_node(node_id, task_desc)
            for n_data in nodes_data:
                target_id = str(n_data["node_id"])
                for source_id in n_data.get("dependency_node_id", []):
                    graph.add_edge(str(source_id), target_id)

        elif schema_b:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)

            edges_data = dag_dict.get("edges", [])
            if not isinstance(edges_data, list):
                edges_data = []
            for e in edges_data:
                if isinstance(e, (list, tuple)) and len(e) == 2:
                    source_id, target_id = e
                    graph.add_edge(str(source_id), str(target_id))

        else:
            raise ValueError("unsupported node schema")

        if len(graph.nodes) == 0:
            raise ValueError("no valid nodes after parsing")

        print(f"✅ Successfully decomposed query into a DAG with {len(graph.nodes)} nodes.")
        return graph

    except Exception as e:
        return _fallback_graph(f"graph build error: {type(e).__name__}: {e}")

In [12]:
def _extract_answer_tag(text: str) -> str:
    """Extract concise answer from <answer>...</answer>. If missing, fallback to stripped text."""
    if text is None:
        return ""
    s = str(text)
    matches = re.findall(r"<answer>\s*(.*?)\s*</answer>", s, flags=re.IGNORECASE | re.DOTALL)
    if matches:
        return matches[-1].strip()
    return s.strip()

def _normalize_answer_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    # Keep only alnum + CJK for robust matching
    s = re.sub(r"[^\w\u4e00-\u9fff]", "", s)
    return s

def score_final_answer_by_gt(client, ground_truth_answer: str, final_output: str) -> int:
    """Scorer-1: use GRADER_MODEL_NAME to judge whether final answer is correct (0/1)."""
    judge_prompt = (
        "You are a strict grader. Determine whether the [Model Final Answer] is semantically equivalent "
        "to the [Ground Truth Answer]. Output 1 if correct, 0 if incorrect. "
        "ignore any formatting differences"
        "Only output a single character: 0 or 1. No explanation.\n\n"
        f"[Ground Truth Answer]\n{ground_truth_answer}\n\n"
        f"[Model Final Answer]\n{final_output}\n"
    )

    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=5,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\b([01])\b", score_str)
        if m:
            return int(m.group(1))

        # Fallback: parse numeric-like output
        m2 = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        if m2:
            return 1 if float(m2.group(1)) >= 0.5 else 0
    except Exception as e:
        print(f"⚠️ Final grading failed; fallback rule enabled: {type(e).__name__}: {e}")

    # Final fallback: heuristic string match
    gt = _normalize_answer_text(ground_truth_answer)
    pred = _normalize_answer_text(final_output)
    if not gt or not pred:
        return 0
    if gt == pred or gt in pred or pred in gt:
        return 1
    return 0

def score_node_by_judge_llm(client, node_input_text: str, node_output: str) -> float:
    """Scorer-2: grade node output right after execution (0~1)."""
    judge_prompt = (
        "You are a strict grader. Based on the [Task Input Text], evaluate how well the [Node Output] "
        "completes the task. Output only a decimal number between 0 and 1 (up to 2 decimals). "
        "No explanation.\n\n"
        f"[Task Input Text]\n{node_input_text}\n\n"
        f"[Node Output]\n{node_output}\n"
    )

    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=10,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        match = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        score = float(match.group(1)) if match else 0.0
    except Exception as e:
        print(f"⚠️ Node grading failed, default to 0: {type(e).__name__}: {e}")
        score = 0.0

    return float(max(0.0, min(1.0, score)))

def build_node_input_text(node: DAGNode) -> str:
    """Build model input text using current node task + predecessor concise answers + node position."""
    predecessor_qa = []
    for p in node.predecessors:
        concise_ans = p.answer_content if getattr(p, "answer_content", "") else _extract_answer_tag(p.output_content)
        predecessor_qa.append(
            f"Predecessor Node {p.node_id}\n"
            f"Question: {p.task_description}\n"
            f"Answer: {concise_ans}"
        )
    predecessor_qa_text = "\n\n".join(predecessor_qa) if predecessor_qa else "No predecessor nodes."

    input_text = (
        f"Node Topological Layer: {node.layer}\n"  # 对应 Algorithm 1 的 node position logic
        f"Current Node Task: {node.task_description}\n\n"
        f"Predecessor Q&A:\n{predecessor_qa_text}\n\n"
        "You may include reasoning, but you MUST provide the final concise answer wrapped in <answer>...</answer>. "
        "Only the content inside <answer> will be used by downstream nodes and grading."
    )
    return input_text

def _execute_single_node_attempt(node: DAGNode, model_input_text: str, chosen_model: str, scores_info: dict, client):
    llm_input_messages = [
        {"role": "user", "content": model_input_text}
    ]

    try:
        resp = client.chat.completions.create(
            model=chosen_model,
            messages=llm_input_messages,
            max_tokens=768,
            temperature=0.1,
        )
        output_content = (resp.choices[0].message.content or "").strip()
    except Exception as e:
        print(f"⚠️ Node {node.node_id} model call failed: {type(e).__name__}: {e}")
        output_content = ""

    answer_content = _extract_answer_tag(output_content)
    judge_target = answer_content if answer_content else output_content
    judge_score = score_node_by_judge_llm(client, model_input_text, judge_target)

    return {
        "chosen_model": chosen_model,
        "ucb_scores": scores_info,
        "output": output_content,
        "parsed_answer": answer_content,
        "judge_score": judge_score,
        "llm_input_messages": llm_input_messages,
    }

def execute_and_evaluate_dag(graph: DAGGraph, problem_desc: str, ground_truth_answer: str,
                             ucb_model, feature_extractor, client):
    """
    Execute DAG nodes, judge each node immediately, and backtrack to the next-best UCB model when needed.
    """
    judge_threshold = float(globals().get("NODE_JUDGE_THRESHOLD", 0.5))
    max_backtracks = int(globals().get("MAX_NODE_BACKTRACKS", 2))

    layers_dict = graph.compute_layers()
    sorted_layers = sorted(layers_dict.keys())
    execution_attempts = []

    # 1) Execute node by node with immediate judge + backtracking
    for layer in sorted_layers:
        for node in layers_dict[layer]:
            model_input_text = build_node_input_text(node)
            node.input_context = model_input_text

            vector_source_text = model_input_text
            x_tn = feature_extractor.get_feature_vector(vector_source_text)
            node.feature_vector = x_tn

            tried_models = []
            final_attempt = None

            for node_attempt_idx in range(max_backtracks + 1):
                chosen_model, scores_info = ucb_model.select_model(x_tn, excluded_models=tried_models)
                if not chosen_model:
                    print(f"⚠️ Node {node.node_id} has no remaining candidate models to try.")
                    break

                attempt_result = _execute_single_node_attempt(
                    node=node,
                    model_input_text=model_input_text,
                    chosen_model=chosen_model,
                    scores_info=scores_info,
                    client=client,
                )
                tried_models.append(chosen_model)
                final_attempt = attempt_result

                execution_attempts.append({
                    "step": len(execution_attempts) + 1,
                    "node_id": node.node_id,
                    "predecessor_node_ids": [p.node_id for p in node.predecessors],
                    "task_desc": node.task_description,
                    "node_attempt_index": node_attempt_idx + 1,
                    "backtrack_round": node_attempt_idx,
                    "chosen_model": chosen_model,
                    "tried_models": list(tried_models),
                    "ucb_scores": scores_info,
                    "output": attempt_result["output"],
                    "parsed_answer": attempt_result["parsed_answer"],
                    "llm_input_messages": attempt_result["llm_input_messages"],
                    "vector_source_text": vector_source_text,
                    "judge_score": float(attempt_result["judge_score"]),
                    "reward": float(attempt_result["judge_score"]),
                })

                if attempt_result["judge_score"] >= judge_threshold:
                    break

                if node_attempt_idx < max_backtracks:
                    print(
                        f"↩️ Node {node.node_id} judge_score={attempt_result['judge_score']:.2f} < "
                        f"{judge_threshold:.2f}, retry with next-best UCB model."
                    )

            if final_attempt is None:
                node.selected_model = None
                node.output_content = ""
                node.answer_content = ""
                node.reward = 0.0
            else:
                node.selected_model = final_attempt["chosen_model"]
                node.output_content = final_attempt["output"]
                node.answer_content = final_attempt["parsed_answer"]
                node.reward = float(final_attempt["judge_score"])

    # 2) Aggregate final output using concise extracted answers from terminal nodes
    terminal_nodes = [n for n in graph.nodes.values() if not n.successors]
    final_output = "\n".join([n.answer_content for n in terminal_nodes]).strip()

    # 3) Final answer grading against ground truth
    final_correct = score_final_answer_by_gt(client, ground_truth_answer, final_output)
    final_status = "success" if final_correct == 1 else "failure"

    # 4) Build observations using the final kept attempt for each node
    observations = []
    for node in graph.nodes.values():
        if node.selected_model is None:
            continue
        observations.append((node.feature_vector, node.selected_model, node.reward))

    return observations, final_output, final_status, execution_attempts, int(final_correct)

In [13]:
import numpy as np
import json
import os

class GreedyLinUCBModel:
    def __init__(
        self,
        model_names,
        feature_dim=EMBEDDING_DIM,
        reg=REG,           
        alpha=GEMMA,       
        seed=42,
    ):
        self.model_names = model_names
        self.feature_dim = feature_dim
        self.reg = reg
        self.alpha = alpha
        
        self.rng = np.random.default_rng(seed)
        self.t = 0
        
        self.models = {
            model_name: {
                "A": np.identity(feature_dim, dtype=np.float32) * self.reg,
                "b": np.zeros(feature_dim, dtype=np.float32)
            } for model_name in self.model_names
        }

    def update(self, feature_vector, model_name, reward):
        """
        更新单个节点的 A 和 b 矩阵。
        """
        if model_name not in self.models:
            return
            
        self.t += 1
        
        # 确保 x 是一维向量
        x = np.asarray(feature_vector, dtype=np.float32).flatten()
        r = float(reward)
        
        # 🟢 实时原地累加，极其轻量
        model = self.models[model_name]
        model["A"] += np.outer(x, x)
        model["b"] += x * r

    def add_observations_and_update(self, observations):
        """
        对应 Algorithm 1 第 20-25 行: Update Models
        遍历 DAG 中每个节点的 observation，更新 A 和 b 矩阵
        """
        for feature_vector, model_name, reward in observations:
            self.update(feature_vector, model_name, reward)

    def select_model(self, feature_vector, excluded_models=None):
        x = np.asarray(feature_vector, dtype=np.float32).flatten()
        excluded_models = set(excluded_models or [])
        best_model = None
        best_score = -float('inf')
        ucb_scores = {}
        
        for name in self.model_names:
            model = self.models[name]
            A = model["A"]
            b = model["b"]
            
            try:
                L = np.linalg.cholesky(A)
                theta = np.linalg.solve(A, b)
                z = np.linalg.solve(L, x)
                
                expected_reward = float(x.dot(theta))
                ucb_term = float(self.alpha * np.sqrt(np.sum(z**2)))
                
            except np.linalg.LinAlgError:
                try:
                    A_inv = np.linalg.inv(A)
                    theta = A_inv.dot(b)
                    
                    expected_reward = float(x.dot(theta))
                    ucb_term = float(self.alpha * np.sqrt(x.dot(A_inv).dot(x)))
                except Exception:
                    expected_reward = 0.0
                    ucb_term = 0.0
            
            total_score = expected_reward + ucb_term
            
            ucb_scores[name] = {
                "pred_score": round(expected_reward, 4),
                "bonus": round(ucb_term, 4),
                "total_ucb": round(total_score, 4),
            }
            
            if name in excluded_models:
                continue

            if total_score > best_score:
                best_score = total_score
                best_model = name
                
        return best_model, ucb_scores

    def save_model_state(self, file_path):
        if not file_path.endswith('.npz'):
            file_path = file_path.rsplit('.', 1)[0] + '.npz'
        save_dict = {f'A_{name}': model['A'] for name, model in self.models.items()}
        for name, model in self.models.items():
            save_dict[f'b_{name}'] = model['b']
        np.savez_compressed(file_path, **save_dict)

    def load_model_state(self, file_path):
        if not file_path.endswith('.npz'):
            file_path = file_path.rsplit('.', 1)[0] + '.npz'
        if not os.path.exists(file_path):
            print(f"⚠️ {file_path} 不存在，将使用全新初始化的 LinUCB 矩阵。")
            return False
        try:
            loaded = np.load(file_path)
            for name in self.models.keys():
                if f'A_{name}' in loaded and f'b_{name}' in loaded:
                    self.models[name]['A'] = loaded[f'A_{name}']
                    self.models[name]['b'] = loaded[f'b_{name}']
            print(f"✅ 成功加载 LinUCB 模型状态: {file_path}")
            return True
        except Exception as e:
            print(f"⚠️ 加载模型状态失败: {e}")
            return False

In [14]:
# # 在 fused_qa_500 中抽取 20 题，测试 Decompose_MODEL_NAME 的分解能力
# # 输入问题字段使用: problem

# from pathlib import Path


# def _find_fused_path() -> Path:
#     candidates = [
#         Path.cwd() / "data" / "fused_qa_500.json",
#         Path.cwd().parent / "data" / "fused_qa_500.json",
#     ]
#     for p in candidates:
#         if p.exists():
#             return p
#     raise FileNotFoundError("未找到 data/fused_qa_500.json")


# fused_path = _find_fused_path()
# with open(fused_path, "r", encoding="utf-8") as f:
#     fused_data = json.load(f)

# assert isinstance(fused_data, list), "fused_qa_500.json 应为 list 结构"
# assert "decompose_query_to_dag" in globals(), "请先运行定义 decompose_query_to_dag 的单元。"

# N_TEST = 20
# selected = []
# for i, rec in enumerate(fused_data):
#     q = rec.get("problem", "")
#     if isinstance(q, str) and q.strip():
#         pid = str(rec.get("problem_id", i))
#         selected.append((pid, q.strip(), rec.get("subject", "unknown")))
#     if len(selected) >= N_TEST:
#         break

# print(f"Using Decompose model: {Decompose_MODEL_NAME}")
# print(f"Dataset: {fused_path.name}")
# print(f"Selected: {len(selected)}\n")

# all_results = []

# for idx, (pid, question, subject) in enumerate(selected, start=1):
#     print("=" * 80)
#     print(f"[{idx}/{len(selected)}] problem_id={pid} | subject={subject}")
#     print(f"Question: {question[:220]}{'...' if len(question) > 220 else ''}")

#     try:
#         graph = decompose_query_to_dag(question, pid, client)
#         node_count = len(graph.nodes)
#         edge_count = sum(len(n.successors) for n in graph.nodes.values())

#         print(f"Decomposition: nodes={node_count}, edges={edge_count}")
#         print("--- Nodes ---")
#         for nid, node in graph.nodes.items():
#             pred_ids = [p.node_id for p in node.predecessors]
#             succ_ids = [s.node_id for s in node.successors]
#             print(f"- {nid}: preds={pred_ids}, succs={succ_ids}")
#             print(f"  task: {node.task_description}")

#         all_results.append({
#             "problem_id": pid,
#             "subject": subject,
#             "ok": True,
#             "nodes": node_count,
#             "edges": edge_count,
#         })
#     except Exception as e:
#         print(f"❌ Failed: {type(e).__name__}: {e}")
#         all_results.append({
#             "problem_id": pid,
#             "subject": subject,
#             "ok": False,
#             "nodes": 0,
#             "edges": 0,
#             "error": str(e),
#         })

# print("\n" + "=" * 80)
# ok_n = sum(int(r["ok"]) for r in all_results)
# avg_nodes = (sum(r["nodes"] for r in all_results if r["ok"]) / ok_n) if ok_n else 0.0
# print(f"Summary: success={ok_n}/{len(all_results)}, avg_nodes={avg_nodes:.2f}")

In [15]:
def main_training_loop(dataset: list, ucb_model, feature_extractor, client, recorder):
    """
    算法第 2-19 行主控流程（含执行、两级评分、参数更新）。
    """
    for idx, record in enumerate(tqdm(dataset, desc="Processing Queries")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = record.get("problem_id", record.get("id", str(idx)))
        gt_answer = record.get("answer", record.get("answers", ""))

        if not raw_query:
            continue

        # 1) 动态分解
        graph = decompose_query_to_dag(raw_query, str(problem_id), client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        # 2) 图执行与评估
        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            ground_truth_answer=gt_answer,
            ucb_model=ucb_model,
            feature_extractor=feature_extractor,
            client=client,
        )

        # 3) 记录
        recorder.add_record(
            problem_id=str(problem_id),
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        # 4) 更新 UCB 参数
        ucb_model.add_observations_and_update(obs)

        print(f"\n[problem_id={problem_id}] final_correct={final_correct}, final_status={final_status}")
        print(f"final_output: {final_out[:200]}{'...' if len(final_out) > 200 else ''}")

    print("🎉 所有查询处理并训练完毕！")


# ==========================================
# 🚀 启动执行模块
# ==========================================
# 1. 实例化核心组件 (已修正为 LinUCB)
ucb_model = GreedyLinUCBModel(model_names=AVAILABLE_LLMS)
recorder = ResultRecorder("execution_records.json")

# 2. 读取数据集
my_dataset = load_dataset("../data/final_evaluation_dataset_500.json")

# # 3. 运行（默认先小样本 smoke test）
# main_training_loop(my_dataset[:2], ucb_model, extractor, client, recorder)
# main_training_loop(my_dataset, ucb_model, extractor, client, recorder)

✅ 成功加载数据集: ..\data\final_evaluation_dataset_500.json


In [16]:
# 小批量测试：使用 5 个问题快速验证端到端流程
SMOKE_N = 5

# 若 extractor 尚未成功初始化，则在此重试一次
if "extractor" not in globals() or extractor is None:
    print("⚠️ extractor 不存在，正在尝试初始化...")
    extractor = FeatureExtractor()

# 准备组件（与主训练流程一致）
ucb_model = GreedyLinUCBModel(model_names=AVAILABLE_LLMS)
recorder = ResultRecorder("execution_records_smoke_5.json")

# 加载数据并截取前 5 条
dataset_path = "../data/final_evaluation_dataset.json"
my_dataset = load_dataset(dataset_path)
small_batch = my_dataset[:SMOKE_N] if isinstance(my_dataset, list) else []

print(f"🚀 Start smoke test with {len(small_batch)} samples")
main_training_loop(small_batch, ucb_model, extractor, client, recorder)
print("✅ Smoke test finished.")

✅ 成功加载数据集: e:\code\321\DAG_Router_demo-gui\data\final_evaluation_dataset_500.json
🚀 Start smoke test with 5 samples


Processing Queries:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Successfully decomposed query into a DAG with 1 nodes.


Processing Queries:  20%|██        | 1/5 [00:10<00:40, 10.15s/it]


[problem_id=1] final_correct=1, final_status=success
final_output: Unbreakable
✅ Successfully decomposed query into a DAG with 3 nodes.
↩️ Node 3 judge_score=0.00 < 0.50, retry with next-best UCB model.
↩️ Node 3 judge_score=0.00 < 0.50, retry with next-best UCB model.


...


[problem_id=2] final_correct=0, final_status=failure
final_output: G
✅ Successfully decomposed query into a DAG with 3 nodes.
↩️ Node 1 judge_score=0.00 < 0.50, retry with next-best UCB model.
↩️ Node 2 judge_score=0.00 < 0.50, retry with next-best UCB model.


Processing Queries:  60%|██████    | 3/5 [01:37<01:09, 34.98s/it]


[problem_id=3] final_correct=1, final_status=success
final_output: J


Processing Queries:  60%|██████    | 3/5 [01:47<01:11, 35.71s/it]


KeyboardInterrupt: 

In [ ]:
from datetime import datetime
from pathlib import Path
import json

def _get_record_dir() -> Path:
    """优先使用已有 record 目录；若不存在则在当前工作目录创建。"""
    candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    p = Path.cwd() / "record"
    p.mkdir(parents=True, exist_ok=True)
    return p

def _resolve_record_path(name_or_path: str, record_dir: Path) -> str:
    p = Path(name_or_path)
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    return str(record_dir / p.name)

def train_with_controls(
    dataset_path: str = "../data/final_evaluation_dataset_500.json",
    train_size: int | None = None,
    use_previous_params: bool = True,
    state_file: str = "ucb_model_state_latest.pt",
    raw_record_file: str = "execution_records_controlled.json",
    per_question_report_file: str = "per_question_metrics.json",
):
    if "extractor" not in globals() or extractor is None:
        print("⚠️ extractor 不存在，正在尝试初始化...")
        globals()["extractor"] = FeatureExtractor()

    record_dir = _get_record_dir()
    state_path = _resolve_record_path(state_file, record_dir)
    raw_record_path_name = Path(raw_record_file).name
    report_path = _resolve_record_path(per_question_report_file, record_dir)

    dataset = load_dataset(dataset_path)
    if not isinstance(dataset, list) or len(dataset) == 0:
        raise ValueError("数据集为空或格式不正确，无法训练。")

    if train_size is not None:
        train_size = max(0, int(train_size))
        dataset = dataset[:train_size]

    if len(dataset) == 0:
        raise ValueError("train_size 截断后数据为空，请调整 train_size。")

    # 实例化已修正为 LinUCB
    ucb_model_local = GreedyLinUCBModel(model_names=AVAILABLE_LLMS)

    if use_previous_params and Path(state_path).exists():
        try:
            ucb_model_local.load_model_state(state_path)
            print(f"🔁 已加载历史参数: {state_path}")
        except Exception as e:
            print(f"⚠️ 历史参数加载失败，将从头训练: {type(e).__name__}: {e}")
    else:
        print("🆕 从随机初始化参数开始训练。")

    recorder_local = ResultRecorder(raw_record_path_name)
    per_question_metrics = []

    for idx, record in enumerate(tqdm(dataset, desc="Controlled Training")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = str(record.get("problem_id", record.get("id", idx)))
        gt_answer = record.get("answer", record.get("answers", ""))

        if not raw_query:
            continue

        graph = decompose_query_to_dag(raw_query, problem_id, client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            ground_truth_answer=gt_answer,
            ucb_model=ucb_model_local,
            feature_extractor=extractor,
            client=client,
        )

        recorder_local.add_record(
            problem_id=problem_id,
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        ucb_model_local.add_observations_and_update(obs)

        terminal_node_ids = [n.node_id for n in graph.nodes.values() if not n.successors]
        attempt_map = {str(a.get("node_id")): a for a in attempts}
        answer_models = [attempt_map[nid].get("chosen_model") for nid in terminal_node_ids if nid in attempt_map]
        used_models = sorted({a.get("chosen_model") for a in attempts if a.get("chosen_model")})

        per_question_metrics.append({
            "problem_id": problem_id,
            "is_correct": int(final_correct),
            "final_status": final_status,
            "answer_model": answer_models[0] if len(answer_models) == 1 else answer_models,
            "used_models": used_models,
            "steps_taken": len(attempts),
            "expected_answer": gt_answer,
            "final_output": final_out,
        })

    ucb_model_local.save_model_state(state_path)

    total = len(per_question_metrics)
    correct = sum(x["is_correct"] for x in per_question_metrics)
    accuracy = (correct / total) if total else 0.0

    report_payload = {
        "config": {
            "dataset_path": dataset_path,
            "train_size": train_size if train_size is not None else "all",
            "use_previous_params": bool(use_previous_params),
            "state_file": state_path,
            "raw_record_file": str(_get_record_dir() / raw_record_path_name),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
        },
        "summary": {
            "total_questions": total,
            "correct_questions": correct,
            "accuracy": round(accuracy, 4),
        },
        "per_question": per_question_metrics,
    }

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report_payload, f, ensure_ascii=False, indent=2)

    print("\n✅ 训练完成")
    print(f"- 准确率: {correct}/{total} = {accuracy:.2%}")
    print(f"- 参数文件: {state_path}")
    print(f"- 逐题报告: {report_path}")
    print(f"- 详细执行记录: {str(_get_record_dir() / raw_record_path_name)}")

    globals()["ucb_model"] = ucb_model_local
    return report_payload

#===== 使用示例 =====
report = train_with_controls(
    dataset_path="../data/final_evaluation_dataset_500.json",
    train_size=100,                 
    use_previous_params=False,      
    state_file="l_ucb_model_state_latest.pt",
    raw_record_file="l_execution_records_train_50.json",
    per_question_report_file="l_per_question_metrics_train_50.json",
)
report["summary"]